In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [5]:
np.random.seed(42)
n_samples = 2000

data = pd.DataFrame({
    'age': np.random.randint(18, 60, n_samples),
    'time_on_site_mins': np.random.uniform(1, 45, n_samples),
    'pages_visited': np.random.randint(1, 20, n_samples),
    'email_opened': np.random.randint(0, 2, n_samples),
    'past_purchases': np.random.randint(0, 5, n_samples)
})

In [7]:
probability = (data['time_on_site_mins'] / 45 * 0.35 + 
               data['pages_visited'] / 20 * 0.3 + 
               data['email_opened'] * 0.2 +
               data['past_purchases'] / 5 * 0.15)

In [9]:
data['converted'] = (probability + np.random.normal(0, 0.1, n_samples) > 0.5).astype(int)

print("Dataset Preview:")
display(data.head())

Dataset Preview:


,age,time_on_site_mins,pages_visited,email_opened,past_purchases,converted
0,56,30.969186,17,0,2,1
1,46,8.983287,4,1,1,0
2,32,24.107189,13,0,3,0
3,25,32.198036,4,1,2,1
4,38,5.702585,5,1,2,0


In [13]:
X = data.drop('converted', axis=1)
y = data['converted']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
print("\nTraining XGBoost Model...")
model = xgb.XGBClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=4, 
    eval_metric='logloss'
)
model.fit(X_train, y_train)


Training XGBoost Model...


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [17]:
preds = model.predict(X_test)
print("\nModel Accuracy:", accuracy_score(y_test, preds))
print("\nClassification Report:\n", classification_report(y_test, preds))


Model Accuracy: 0.8475

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.84      0.84       197
           1       0.85      0.85      0.85       203

    accuracy                           0.85       400
   macro avg       0.85      0.85      0.85       400
weighted avg       0.85      0.85      0.85       400



In [19]:
joblib.dump(model, 'lead_scoring_model.pkl')
print("Model saved successfully as 'lead_scoring_model.pkl'")

Model saved successfully as 'lead_scoring_model.pkl'


In [21]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import pandas as pd

In [25]:
app = FastAPI(
    title="Lead Scoring Prediction API",
    description="An API that evaluates customer data and predicts conversion probability.",
    version="1.0"
)

In [27]:
model = joblib.load('lead_scoring_model.pkl')

In [29]:
class LeadData(BaseModel):
    age: int
    time_on_site_mins: float
    pages_visited: int
    email_opened: int
    past_purchases: int

In [40]:
@app.post("/predict")
def score_lead(lead: LeadData):
    input_df = pd.DataFrame([lead.dict()])

    prediction = model.predict(input_df)[0]
    probability = model.predict_proba(input_df)[0][1]

    return {
        "will_convert": bool(prediction),
        "conversion_probability": round(float(probability) * 100, 2),
        "recommendation": "Prioritize Lead" if probability > 0.5 else "Nurture Lead"
    }

In [49]:
!uvicorn app:app --reload --host 0.0.0.0 --port 8000

^C


In [52]:
import os
print(os.path.exists("app.py"))

False


In [54]:
%%writefile app.py
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import pandas as pd

app = FastAPI(
    title="Lead Scoring Prediction API",
    description="An API that evaluates customer data and predicts conversion probability.",
    version="1.0"
)

model = joblib.load('lead_scoring_model.pkl')

class LeadData(BaseModel):
    age: int
    time_on_site_mins: float
    pages_visited: int
    email_opened: int
    past_purchases: int

@app.post("/predict")
def score_lead(lead: LeadData):
    input_df = pd.DataFrame([lead.dict()])
    
    prediction = model.predict(input_df)[0]
    probability = model.predict_proba(input_df)[0][1]
    
    return {
        "will_convert": bool(prediction),
        "conversion_probability": round(float(probability) * 100, 2),
        "recommendation": "Prioritize Lead" if probability > 0.5 else "Nurture Lead"
    }

Writing app.py


In [56]:
import os
print(os.path.exists("app.py"))

True


In [69]:
!uvicorn app:app --reload

^C


In [61]:
%%writefile dashboard.py
import streamlit as st
import requests

st.title("🎯 Automated Lead Scoring Dashboard")
st.write("Input prospective customer behavior metrics to calculate conversion probability in real time.")

col1, col2 = st.columns(2)

with col1:
    age = st.number_input("Customer Age", min_value=18, max_value=90, value=30)
    time_on_site = st.slider("Time on Site (minutes)", 0.0, 60.0, 15.0)
    pages_visited = st.slider("Pages Visited during Session", 1, 30, 5)

with col2:
    email_opened = st.selectbox("Opened Recent Marketing Email?", [0, 1], format_func=lambda x: "Yes" if x == 1 else "No")
    past_purchases = st.number_input("Past Purchases Count", min_value=0, max_value=20, value=0)

if st.button("Analyze & Score Lead", type="primary"):
    payload = {
        "age": int(age),
        "time_on_site_mins": float(time_on_site),
        "pages_visited": int(pages_visited),
        "email_opened": int(email_opened),
        "past_purchases": int(past_purchases)
    }
    
    try:
        response = requests.post("http://127.0.0.1:8000/predict", json=payload)
        result = response.json()
        
        st.divider()
        st.metric(label="Conversion Probability", value=f"{result['conversion_probability']}%")
        
        if result['will_convert']:
            st.success(f"🔥 High Intent Lead — Action: **{result['recommendation']}**")
        else:
            st.warning(f"❄️ Low Intent Lead — Action: **{result['recommendation']}**")
            
    except Exception as e:
        st.error(f"Could not connect to FastAPI server. Ensure uvicorn is running on port 8000. Error: {e}")

Overwriting dashboard.py


In [68]:
!streamlit run dashboard.py

^C


In [72]:
%%writefile requirements.txt
pandas>=2.0.0
numpy>=1.24.0
scikit-learn>=1.2.0
xgboost>=1.7.0
fastapi>=0.95.0
uvicorn>=0.22.0
pydantic>=1.10.0
streamlit>=1.22.0
requests>=2.28.0
joblib>=1.2.0

Writing requirements.txt


In [74]:
%%writefile README.md

An end-to-end Machine Learning solution that predicts customer lead conversion probabilities using **XGBoost**, serves real-time predictions via a **FastAPI** REST API, and provides an interactive sales team dashboard built with **Streamlit**.

---

Sales and marketing teams spend valuable time reaching out to cold leads that never convert, increasing Customer Acquisition Costs (CAC). 

This project automates lead qualification by evaluating prospective customer behavior metrics—such as session duration, page views, email engagement, and purchase history—and assigning each lead a conversion probability score (0–100%) in real time. 

- **Optimized Sales Focus:** Prioritizes leads with >50% conversion probability for immediate sales follow-up.
- **Automated Nurturing:** Identifies low-intent leads for automated email nurturing campaigns.

---


```text
[ Raw Data / CRM ] ──> [ Data Processing & Feature Engineering ]
                                │
                                ▼
                       [ XGBoost Classifier ]
                                │
                                ▼
                   [ FastAPI REST Endpoint ]
                                │
                                ▼
                   [ Streamlit Sales Dashboard ]

Writing README.md
